In [19]:
!pip install transformers accelerate bitsandbytes gradio fpdf pillow gtts

In [ ]:
from huggingface_hub import login
login(token="YOUR_TOKEN")


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "google/gemma-2b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [12]:
import random, uuid, textwrap
from fpdf import FPDF
from PIL import Image, ImageDraw, ImageFont
from gtts import gTTS
import os

def generate_excuse(name, institute, context, reason, urgency, tone):
    prompt = f"""
You are an intelligent excuse generator. Generate a long, detailed, realistic and convincing excuse.

Details:
- Name: {name}
- Institute/Company: {institute}
- Context: {context}
- Reason: {reason}
- Urgency Level: {urgency}
- Tone: {tone}

Also include any believable proof or supporting information in a natural way.

Excuse:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.95,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return result.strip()


In [13]:
def create_pdf(excuse_text, name):
    filename = f"/content/excuse_{uuid.uuid4().hex[:6]}.pdf"
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, f"To Whom It May Concern,\n\n{name} provides the following explanation:\n\n{excuse_text}")
    pdf.output(filename)
    return filename


In [14]:
def create_chat_image(name, excuse_text, context):
    lines = {
        "Work": [f"Boss: Why did you miss the meeting?", f"{name}: {excuse_text}"],
        "School": [f"Teacher: Where is your assignment?", f"{name}: {excuse_text}"],
        "Social": [f"Friend: You didn’t show up yesterday!", f"{name}: {excuse_text}"],
        "Family": [f"Mom: Why didn’t you come home early?", f"{name}: {excuse_text}"]
    }.get(context, [f"Other: What happened?", f"{name}: {excuse_text}"])

    img = Image.new("RGB", (600, 400), color="white")
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    y = 20
    for msg in lines:
        for line in textwrap.wrap(msg, width=70):
            draw.text((20, y), line, font=font, fill="black")
            y += 20
        y += 10
    path = f"/content/chat_{uuid.uuid4().hex[:6]}.png"
    img.save(path)
    return path


In [15]:
def create_certificate(name, institute, context, reason, excuse_text):
    img = Image.new("RGB", (700, 500), color=(255, 255, 240))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    y = 30
    lines = [
        "⚠️ OFFICIAL EXCUSE CERTIFICATE ⚠️", "", f"Name: {name}", f"Institution: {institute}",
        f"Context: {context}", f"Reason: {reason}", "", "This is to certify the following excuse:",
        "", *textwrap.wrap(excuse_text, width=80), "", "Authorized by: AI Excuse Generator"
    ]
    for line in lines:
        draw.text((20, y), line, font=font, fill="black")
        y += 25
    path = f"/content/cert_{uuid.uuid4().hex[:6]}.png"
    img.save(path)
    return path


In [16]:
def create_voice(excuse_text):
    tts = gTTS(excuse_text)
    path = f"/content/voice_{uuid.uuid4().hex[:6]}.mp3"
    tts.save(path)
    return path


In [17]:
def full_generator(name, institute, context, reason, urgency, tone):
    excuse = generate_excuse(name, institute, context, reason, urgency, tone)
    pdf = create_pdf(excuse, name)
    chat = create_chat_image(name, excuse, context)
    cert = create_certificate(name, institute, context, reason, excuse)
    voice = create_voice(excuse)
    return excuse, [pdf, chat, cert, voice]


In [18]:
import gradio as gr

gr.Interface(
    fn=full_generator,
    inputs=[
        gr.Textbox(label="Name"),
        gr.Textbox(label="Institute / Company"),
        gr.Dropdown(["Work", "School", "Social", "Family"], label="Context"),
        gr.Dropdown(["Health", "Personal", "Emergency", "Technical"], label="Reason"),
        gr.Dropdown(["Low", "Medium", "High"], label="Urgency"),
        gr.Dropdown(["Professional", "Emotional", "Funny", "Casual"], label="Tone"),
    ],
    outputs=[
        gr.Textbox(label="Generated Excuse"),
        gr.Files(label="📎 Download: PDF, Chat Image, Certificate, Voice")
    ],
    title="🎭 AI Excuse Generator (Gemma-powered)",
    description="Generate detailed excuses + fake proofs using Google's Gemma model"
).launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f942e2b8b337af9cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
